# P2 · N4 — Downstream Impact, Fairness, and Cost

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

Answers RQ5: does gating training data with machine-authored rules change
downstream model discrimination or subgroup fairness, and at what cost?

**Design.** Gating is a training-data curation decision, so the gate is applied
to the **training partition only**. Every condition is then evaluated on the
**same ungated test partition** — otherwise each condition would be scored on a
different population and the comparison would be meaningless.

```
evaluation split (80%, from N0)
  |-- train 70%   <- gate applied here
  |-- test  30%   <- never gated, identical for every condition
```

**Infeasible gates are a result.** A rule set that empties the training
partition, or removes an entire class, makes fitting impossible. That is
recorded with a reason, never skipped — roughly a sixth of rule sets fall into
this category and dropping them would flatter the rest.

**Prerequisites:** N0–N3 complete.
**Runtime:** 20–40 minutes. Checkpointed; safe to interrupt. No API calls.

## 1 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Setup

In [ ]:
import sys, json, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'
RUNS        = ROOT / 'runs'

assert ROOT.exists() and MODULES.exists()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import llmauth_ir as IR
import llmauth_downstream as DS
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)
RUN_LOG_PATH = RUNS / 'runs.jsonl'
EXCLUDE_MODELS = {'mock-1', 'qwen2.5-7b', 'claude-opus-4-8'}

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv','.parquet','.xlsx')]
    out = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits: out[corpus] = hits[0]
    return out

print('census', C.CENSUS_VERSION, '| ir', IR.SCHEMA_VERSION,
      '| downstream', DS.DOWNSTREAM_VERSION)
print('supervised tasks:', list(DS.TASKS))

## 3 · Tasks and leakage exclusions

Each corpus has features that encode the label. These are excluded by name, and
the exclusions belong in the paper's Methods.

`online_retail_ii` has no natural supervised label and is not analysed here; it
contributed detection, retention, and cost only.

In [ ]:
for cid, t in DS.TASKS.items():
    print(f'{cid}')
    print(f'   target    : {t.target_column}  (positive = '
          f'{"callable" if callable(t.positive) else repr(t.positive)})')
    print(f'   excluded  : {t.drop_columns}')
    print(f'   subgroups : {t.subgroups}')
    if t.max_rows: print(f'   subsample : {t.max_rows:,} rows')
    print(f'   note      : {t.note}\n')

## 4 · Load rule sets from the run log

In [ ]:
records = [json.loads(l) for l in open(RUN_LOG_PATH) if l.strip()]
records = [r for r in records if r['key']['model_id'] not in EXCLUDE_MODELS]
records = [r for r in records if r['key']['corpus_id'] in DS.TASKS]

rulesets = {}
for r in records:
    k = r['key']
    key = (k['corpus_id'], k['condition'], k['model_id'], k['seed'])
    rulesets[key] = IR.parse_llm_response(
        r['raw_response'] or '', corpus_id=k['corpus_id'],
        condition=k['condition'], model_id=k['model_id'], seed=k['seed'])

print(f'{len(rulesets)} rule sets across {len(DS.TASKS)} supervised corpora')
print(pd.Series([k[0] for k in rulesets]).value_counts().to_string())

## 5 · Build features and the train/test partition

Encoding is fitted on the full evaluation split **before** any gating, so every
condition sees an identical feature space. Encoding uses no label information,
so this does not leak.

In [ ]:
DATA_PATHS = discover_data()
prepared = {}

for cid, task in DS.TASKS.items():
    df, _ = C.load_corpus(cid, DATA_PATHS[cid])
    split = ckpt.step(f'split_{cid}', 'json',
                      lambda: (_ for _ in ()).throw(RuntimeError('run N0 first')))[0]
    ev = df.loc[split['evaluation_index']]

    # subsample ONCE so masks and features refer to the same rows
    if task.max_rows and len(ev) > task.max_rows:
        ev = ev.sample(n=task.max_rows, random_state=DS.SPLIT_SEED).sort_index()

    X, y, strata = DS.prepare_features(ev, task)
    tr, te = DS.train_test_split_eval(ev.index)
    prepared[cid] = dict(ev=ev, X=X, y=y, strata=strata, train=tr, test=te)

    print(f'{cid:<18} eval {len(ev):>7,}  train {len(tr):>7,}  test {len(te):>7,}  '
          f'features {X.shape[1]:>2}  positive {y.mean():.4f}')

## 6 · No-gate baselines

Every gated configuration is compared against this. If no gate beats it, that
is the finding.

In [ ]:
def _baselines():
    out = []
    for cid, p in prepared.items():
        r = DS.fit_and_evaluate(p['X'], p['y'], p['strata'], p['train'], p['test'],
                                None, corpus_id=cid, condition='NO_GATE',
                                model_id='baseline', seed=0)
        out.append(r.to_dict())
    return out

baselines, _ = ckpt.step('n4_baselines', 'json', _baselines,
                         code_version=DS.DOWNSTREAM_VERSION)
base_by_corpus = {b['corpus_id']: b for b in baselines}

print(f'{"corpus":<18}{"ROC-AUC":>9}{"PR-AUC":>9}{"Brier":>9}{"train rows":>12}')
print('-'*57)
for b in baselines:
    print(f'{b["corpus_id"]:<18}{b["roc_auc"]:>9.4f}{b["pr_auc"]:>9.4f}'
          f'{b["brier"]:>9.4f}{b["train_rows"]:>12,}')

## 7 · Evaluate every gate

Masks are cached by canonical rule form, so a rule recurring across seeds and
conditions is executed once. Checkpointed — safe to interrupt and re-run.

In [ ]:
mask_cache = {}

def rule_mask(cid, rule):
    key = (cid, IR.canonical_form(rule))
    if key not in mask_cache:
        try:
            mask_cache[key] = IR.to_pandas_mask(rule)(prepared[cid]['ev'])
        except Exception:
            mask_cache[key] = None
    return mask_cache[key]

def gate_mask(cid, rs):
    ev = prepared[cid]['ev']
    keep = pd.Series(True, index=ev.index)
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in ev.columns:
            continue
        m = rule_mask(cid, rule)
        if m is not None:
            keep &= m
    return keep

def _run_all():
    out = []
    items = sorted(rulesets.items())
    for i, (key, rs) in enumerate(items, 1):
        cid, cond, model, seed = key
        p = prepared[cid]
        r = DS.fit_and_evaluate(p['X'], p['y'], p['strata'], p['train'], p['test'],
                                gate_mask(cid, rs), corpus_id=cid, condition=cond,
                                model_id=model, seed=seed)
        out.append(r.to_dict())
        if i % 20 == 0 or i == len(items):
            nf = sum(1 for o in out if not o['feasible'])
            print(f'  {i}/{len(items)}   infeasible so far: {nf}')
    return out

results, cached = ckpt.step('n4_downstream_results', 'json', _run_all,
                            code_version=DS.DOWNSTREAM_VERSION)
res = pd.DataFrame(results)
print(f'\n{len(res)} gates evaluated, {int((~res.feasible).sum())} infeasible')

## 8 · Infeasible gates

A gate that cannot be fitted is a governance failure with a concrete
consequence: the data product cannot be built.

In [ ]:
inf = res[~res.feasible]
print(f'{len(inf)}/{len(res)} gates ({len(inf)/len(res):.1%}) make training impossible\n')
if len(inf):
    print(inf.pivot_table(index='corpus_id', columns='condition',
                          values='seed', aggfunc='count').fillna(0).astype(int).to_string())
    print('\nreasons:')
    print(inf.reason.value_counts().head(8).to_string())

## 9 · Downstream discrimination versus no gate

The headline comparison. `delta_auc` is the gated model's ROC-AUC minus the
no-gate baseline on the same test set. Positive means the gate helped.

In [ ]:
feas = res[res.feasible].copy()
feas['base_auc'] = feas.corpus_id.map(lambda c: base_by_corpus[c]['roc_auc'])
feas['delta_auc'] = feas.roc_auc - feas.base_auc

piv = feas.pivot_table(index=['corpus_id','model_id'], columns='condition',
                       values='delta_auc', aggfunc='mean').round(4)
print('DELTA ROC-AUC vs NO GATE\n'); print(piv.to_string())

print('\n\npooled by condition:')
print(feas.groupby('condition').delta_auc.agg(['mean','std','min','max']).round(4).to_string())

def boot_ci(x, n=10000):
    x = np.asarray(x)
    bs = [np.random.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

print('\n95% bootstrap CI on mean delta_auc, by condition:')
for c in sorted(feas.condition.unique()):
    x = feas[feas.condition==c].delta_auc.values
    lo, hi = boot_ci(x)
    verdict = 'improves' if lo > 0 else ('degrades' if hi < 0 else 'no effect')
    print(f'  {c}: {x.mean():+.4f}  CI [{lo:+.4f}, {hi:+.4f}]   {verdict}')

best = feas.delta_auc.max()
print(f'\nbest single gate: {best:+.4f} AUC vs baseline')
print(f'gates that beat no-gate at all: {(feas.delta_auc>0).sum()}/{len(feas)} '
      f'({(feas.delta_auc>0).mean():.1%})')

## 10 · Subgroup fairness

Two measures. `auc_gap` is the spread in discrimination across levels of a
protected attribute, on the shared test set. `representation_shift` is how much
the gate changed the composition of the training population — half the total
variation distance between the subgroup distribution before and after gating.

In [ ]:
sub = DS.subgroup_report([DS.DownstreamResult(**{
    k: v for k, v in r.items() if k in DS.DownstreamResult.__dataclass_fields__})
    for r in results])
sub = sub[sub.feasible]

print('MEAN SUBGROUP AUC GAP\n')
print(sub.pivot_table(index=['corpus','subgroup'], columns='condition',
                      values='auc_gap', aggfunc='mean').round(4).to_string())

print('\n\nMEAN REPRESENTATION SHIFT (0 = gate did not change composition)\n')
print(sub.pivot_table(index=['corpus','subgroup'], columns='condition',
                      values='representation_shift', aggfunc='mean').round(4).to_string())

worst = sub.sort_values('representation_shift', ascending=False).head(10)
print('\n\nlargest composition shifts:')
print(worst[['corpus','subgroup','condition','model','representation_shift',
             'auc_gap']].to_string(index=False))

## 11 · Cost

Authoring cost alone understates the cost of a bad gate. Cost per million
retained training records charges a rule set for the data it destroys.

In [ ]:
costs = {}
for r in records:
    k = r['key']
    costs[(k['corpus_id'], k['condition'], k['model_id'], k['seed'])] = {
        'cost_usd': r.get('cost_usd') or 0.0,
        'input_tokens': r.get('input_tokens') or 0,
        'output_tokens': r.get('output_tokens') or 0,
        'latency_seconds': r.get('latency_seconds') or 0.0,
    }

feas['cost_usd'] = feas.apply(
    lambda r: costs.get((r.corpus_id, r.condition, r.model_id, r.seed), {}).get('cost_usd', 0.0),
    axis=1)
feas['cost_per_M_retained'] = feas.apply(
    lambda r: DS.cost_per_retained_record(r.cost_usd, r.train_rows), axis=1)

print('MEAN AUTHORING COST (USD per rule set)\n')
print(feas.pivot_table(index='model_id', columns='condition',
                       values='cost_usd', aggfunc='mean').round(4).to_string())

print('\n\nMEAN COST PER MILLION RETAINED TRAINING RECORDS (USD)\n')
print(feas.pivot_table(index=['corpus_id','model_id'], columns='condition',
                       values='cost_per_M_retained', aggfunc='mean').round(3).to_string())

tot = sum(c['cost_usd'] for c in costs.values())
print(f'\ntotal authoring cost across all runs: ${tot:.2f}')
print(f'mean latency per rule set: '
      f'{np.mean([c["latency_seconds"] for c in costs.values()]):.1f}s')

## 12 · Save and provenance

In [ ]:
feas.to_csv(ARTIFACTS / 'N4_downstream.csv', index=False)
sub.to_csv(ARTIFACTS / 'N4_subgroups.csv', index=False)
res.to_csv(ARTIFACTS / 'N4_all_gates.csv', index=False)

prov = {
    'notebook': 'P2_N4_downstream',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'downstream_version': DS.DOWNSTREAM_VERSION,
    'excluded_models': sorted(EXCLUDE_MODELS),
    'test_fraction': DS.TEST_FRACTION,
    'split_seed': DS.SPLIT_SEED,
    'tasks': {c: {'target': t.target_column, 'dropped': t.drop_columns,
                  'subgroups': t.subgroups, 'note': t.note}
              for c, t in DS.TASKS.items()},
    'baselines': {b['corpus_id']: {'roc_auc': b['roc_auc'], 'pr_auc': b['pr_auc'],
                                   'train_rows': b['train_rows']} for b in baselines},
    'n_gates': int(len(res)), 'n_infeasible': int((~res.feasible).sum()),
}
(ARTIFACTS / 'N4_provenance.json').write_text(json.dumps(prov, indent=2, default=str))
print('wrote', ARTIFACTS / 'N4_downstream.csv')
print('wrote', ARTIFACTS / 'N4_provenance.json')
print()
ckpt.status()